[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-association.ipynb)

# Association Rule Mining

*AIBits Academy · Machine Learning End To End · Unsupervised Learning · New*

Market-basket analysis: discovering which items are bought together, and the Apriori and FP-Growth algorithms that make the search tractable.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q mlxtend

> **🎯 Intuition First**
>
> A kirana-store owner in Nagpur notices, without any statistics, that customers who buy *poha* almost always grab *chivda* too, so he moves them to the same shelf and sales rise. Association Rule Mining is that instinct, made rigorous and automatic. It doesn't care *who* the customer is (that's what recommender systems do) — it only asks a question about the baskets themselves: **which items tend to show up in the same transaction?** The output is a set of human-readable rules like "{Bread} → {Butter}", each carrying numbers that tell you how common the pattern is and, crucially, whether it's a *real* association or just an artefact of some item being popular anyway.

> **🔗 Continues from Recommender Systems**
>
> The page ended by contrasting collaborative filtering ("what would *this user* like?") with association rules ("what is bought *together*, regardless of who?"). This page is the full treatment of that second idea.

## The Three Core Metrics

Every association rule X → Y (read "if a basket contains itemset X, it also tends to contain Y") is scored by three numbers built from **support** — the fraction of all transactions containing a given itemset.

$$\begin{gathered}\mathrm{Support}(X) = \dfrac{\text{transactions containing } X}{\text{total transactions}} \\[6pt] \mathrm{Confidence}(X\to Y) = \dfrac{\mathrm{Support}(X\cup Y)}{\mathrm{Support}(X)} = P(Y\mid X) \\[6pt] \mathrm{Lift}(X\to Y) = \dfrac{\mathrm{Confidence}(X\to Y)}{\mathrm{Support}(Y)}\end{gathered}$$

**Support** asks how common the pattern is overall (low-support rules may be statistical noise). **Confidence** asks: given X was bought, how often was Y also bought? **Lift** is the essential quality check — it divides confidence by Y's own baseline popularity. Lift > 1 means X and Y co-occur *more* than chance predicts (a genuine association); Lift ≈ 1 means Y would have been bought about that often anyway; Lift < 1 means buying X actually makes Y *less* likely. This lift correction is what stops you from acting on high-confidence-but-meaningless rules whose consequent is simply a near-universal staple.

## Try It — Support / Confidence / Lift Calculator

A fixed set of 8 grocery baskets over 4 items. Pick an antecedent X (blue) and a consequent Y (orange) to build a rule, and watch all five metrics compute live. Try {Bread}→{Butter} (positive lift), then {Jam}→{Milk} (lift < 1: buying Jam makes Milk *less* likely here).

## Worked Example — {Bread} → {Butter} by Hand

Using those same 8 baskets: Bread appears in 6, Butter appears in 6, and both appear together in 5.

$$\begin{gathered}\mathrm{Support}(\text{Bread}) = \tfrac{6}{8} = 0.750 \qquad \mathrm{Support}(\text{Butter}) = \tfrac{6}{8} = 0.750 \qquad \mathrm{Support}(\text{Bread}\cup\text{Butter}) = \tfrac{5}{8} = 0.625 \\[6pt] \mathrm{Confidence}(\text{Bread}\to\text{Butter}) = \dfrac{0.625}{0.750} = \mathbf{0.833} \\[6pt] \mathrm{Lift}(\text{Bread}\to\text{Butter}) = \dfrac{0.833}{0.750} = \mathbf{1.111} \quad (>1 \to \text{genuine positive association}) \\[6pt] \mathrm{Leverage} = 0.625 - (0.750\times 0.750) = \mathbf{0.062} \qquad \mathrm{Conviction} = \dfrac{1-0.750}{1-0.833} = \mathbf{1.500}\end{gathered}$$

**Leverage** measures how far the co-occurrence exceeds what independence would give (0 = independent). **Conviction** is the ratio of "how often X appears without Y if they were independent" to "how often that actually happens" — higher means the rule is harder to violate, so a stronger implication. All five numbers here are verified against the `mlxtend` library.

## The Combinatorial Problem — and the Apriori Principle

With p distinct items there are 2ᵖ − 1 possible itemsets, and vastly more possible rules — brute-forcing every one is hopeless even for a modest catalogue. The **Apriori principle** (Agrawal & Srikant, 1994) is the insight that tames it:

> **⬇ The Apriori Principle (Downward Closure)**
>
> **If an itemset is infrequent, then every superset of it must also be infrequent.** If {Milk, Jam} appears in too few baskets to clear the support threshold, then {Bread, Milk, Jam} — which can only appear in a *subset* of those baskets — cannot possibly clear it either. So the moment an itemset is pruned, its entire "upward" branch of the lattice is pruned too, without ever computing their supports.

## See the Pruning — Interactive Itemset Lattice

The full lattice of all 15 itemsets over 4 items, with each node's true support (from the same 8 baskets). Drag the `min_support` threshold up and watch itemsets fall below it get pruned (red, struck through) — *and every superset above them prune automatically*, exactly as the Apriori principle guarantees. Blue = frequent (kept for rule generation); red = infrequent (pruned). The kept-count at each threshold is verified against `mlxtend`'s `apriori()`.

## Apriori in Python (mlxtend)

The Apriori algorithm has two phases: (1) use the pruning principle above to find all *frequent itemsets* clearing `min_support`, then (2) generate rules from those itemsets that clear a `min_confidence` threshold. On an Ahmedabad kirana store's baskets:

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd

# Each row is one customer's basket
transactions = [
    ['Rice', 'Dal', 'Ghee'],
    ['Rice', 'Dal', 'Milk'],
    ['Bread', 'Milk', 'Butter'],
    ['Rice', 'Dal', 'Ghee', 'Milk'],
    ['Bread', 'Butter'],
    ['Rice', 'Ghee'],
]
items = sorted(set(i for t in transactions for i in t))
onehot = pd.DataFrame([{item: (item in t) for item in items} for t in transactions])

frequent_itemsets = apriori(onehot, min_support=0.3, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
print(rules[['antecedents','consequents','support','confidence','lift']]
      .sort_values('lift', ascending=False).head(4))

Bread → Butter has the highest lift (3.0): bread-buyers are 3× more likely to also buy butter than a random customer — a strong, actionable association (shelf placement, bundle discount). Dal → Rice has lift 1.5 — a real but weaker association, unsurprising since both are staples appearing in many baskets anyway.

## FP-Growth — Skipping Candidate Generation

Apriori's weakness is that it makes **multiple passes over the data** and explicitly generates candidate itemsets at each level — expensive on large transaction databases. **FP-Growth** (Frequent-Pattern Growth) avoids both. It compresses the entire dataset into a compact **FP-tree** (a prefix tree of items ordered by frequency, so common items are shared near the root), then mines frequent itemsets by recursively walking that tree — *no candidate generation, and only two passes over the raw data*. On the identical kirana dataset it returns the *exact same* frequent itemsets and rules as Apriori — only faster:

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

# onehot is exactly the same DataFrame built for Apriori above
frequent_itemsets = fpgrowth(onehot, min_support=0.3, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
print(rules[['antecedents','consequents','support','confidence','lift']]
      .sort_values('lift', ascending=False).head(4))

The identical Bread ↔ Butter rules top the list (order among equal-lift rules can differ, since FP-Growth enumerates itemsets in a different sequence, but the rule *set* and every metric is identical to Apriori's). The rule of thumb: use Apriori to learn the concept and for small datasets; reach for FP-Growth (or the even faster **ECLAT**) when transaction volumes grow large.

## Apriori vs FP-Growth

|  | Apriori | FP-Growth |
|---|---|---|
| Core idea | Generate & test candidate itemsets level by level | Compress to an FP-tree, mine recursively |
| Candidate generation | Yes — explicit, at every level | None |
| Passes over data | Many (one per level) | Two |
| Speed on large data | Slower | Substantially faster |
| Memory | Low | Higher (holds the FP-tree) |
| Output | Identical frequent itemsets & rules |  |

## Collaborative Filtering vs Association Rules

|  | Collaborative Filtering | Association Rule Mining |
|---|---|---|
| Answers | "What would *this specific user* like?" | "What tends to be bought *together*, regardless of who?" |
| Needs user history? | Yes — builds a taste profile per user | No — works purely at the transaction/basket level |
| Typical use | Homepage personalisation, "recommended for you" | "Frequently bought together," shelf/catalogue placement, bundle design |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Support and confidence by hand

From `baskets`, compute `support_rd` = support of {Rice, Dal} (share of baskets containing both) and `conf` = confidence of the rule Rice → Dal (baskets with both / baskets with Rice).

In [ ]:
baskets = [{"Rice", "Dal"}, {"Rice", "Dal", "Ghee"}, {"Bread", "Milk"}, {"Rice", "Milk"}, {"Rice", "Dal", "Milk"}, {"Bread", "Butter"}]
support_rd = conf = None   # TODO


In [ ]:
try:
    check("support 3/6", support_rd == 0.5)
    check("confidence 3/4", conf == 0.75)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
baskets = [{"Rice", "Dal"}, {"Rice", "Dal", "Ghee"}, {"Bread", "Milk"}, {"Rice", "Milk"}, {"Rice", "Dal", "Milk"}, {"Bread", "Butter"}]
both = sum({"Rice", "Dal"} <= b for b in baskets)
rice = sum("Rice" in b for b in baskets)
support_rd = both / len(baskets)
conf = both / rice

```

</details>

### Exercise 2 · Medium · Lift

Lift = confidence / support of the consequent. Store in `lift` the lift of Rice → Dal using the baskets above. A lift above 1 means the items appear together more than chance predicts.

In [ ]:
lift = None   # TODO (reuse baskets, conf)


In [ ]:
try:
    check("lift = 0.75 / 0.5 = 1.5", abs(lift - 0.75 / 0.5) < 1e-12)
    check("greater than 1", lift > 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
dal = sum("Dal" in b for b in baskets) / len(baskets)
lift = conf / dal

```

</details>

### Exercise 3 · Stretch · Mine rules with Apriori

One-hot encode `baskets` with `TransactionEncoder`, run `apriori(min_support=0.3, use_colnames=True)` and `association_rules(metric="lift", min_threshold=1.0)`. Store the rules in `rules` and the number of frequent itemsets in `n_sets`.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
rules = n_sets = None   # TODO


In [ ]:
try:
    check("some rules found", len(rules) > 0)
    check("all have lift >= 1", (rules["lift"] >= 1).all())
    check("itemsets counted", n_sets >= 5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
te = TransactionEncoder()
onehot = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)
freq = apriori(onehot, min_support=0.3, use_colnames=True)
n_sets = len(freq)
rules = association_rules(freq, metric="lift", min_threshold=1.0)

```

Support prunes rare itemsets, confidence measures reliability, lift removes rules that merely reflect popular items.

</details>

---
*Back to the course: **Machine Learning End To End → Association Rule Mining**.*